In [0]:
from pyspark.sql.functions import col, concat_ws, explode_outer, from_json, lower, upper, current_timestamp, lit, regexp_replace, to_date,concat, current_date, expr
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, IntegerType
from delta.tables import DeltaTable

# ============ CONFIGURAÇÕES ============
catalogo = "`oc_projeto`"
schema = "`oc-tabelas`"
table_bronze = f'{catalogo}.{schema}.flights_bronze'
table_silver = f'{catalogo}.{schema}.silver_voos_arr'


In [0]:
# ============ PASSO 1: LER BRONZE ============
print("\nPASSO 1: Lendo BRONZE...\n")

# lê a tabela bronze com dados de 7 dias anteriores
df_bronze = spark.table(table_bronze).filter(
    col("data_carregamento") >= current_date() - expr("INTERVAL 7 DAYS"))

In [0]:

# ============ PASSO 2: EXPLODIR VOOS_RECEBIDOS ============
print("\nPASSO 2: Explodindo voos_recebidos...\n")

df_recebidos = df_bronze.select(
    col("turno"),
    col("data"),
    col("agente"),
    explode_outer(col("voos_recebidos")).alias("voo_recebido"),
    lit(table_bronze).alias("tabela_origem"),
    col("data_carregamento")
)


In [0]:
# ============ PASSO 3: DESANINHAR VOO_RECEBIDO ============
print("\n PASSO 3: Desaninhando voo_recebido...\n")

df_recebidos_flat = df_recebidos.select(
    col("turno"),
    col("data"),
    col("agente"),
    col("voo_recebido.*"),
    col("tabela_origem"),
    col("data_carregamento")
)
print(f"* Estrutura desaninhada")


In [0]:
# ============ PASSO 4: CONVERTER AF_CHEGANDO DE STRING PARA STRUCT ============
print("\n PASSO 4: Convertendo AF_chegando (JSON string → Struct)...\n")

af_chegando_schema = StructType([
    StructField("origem", StringType()),
    StructField("destinos", ArrayType(StructType([
        StructField("voo", StringType()),
        StructField("destino", StringType()),
        StructField("qtd", IntegerType())
    ])))
])

df_recebidos_com_af = df_recebidos_flat.select(
    col("turno"),
    col("data"),
    col("agente"),
    col("voo"),
    col("iata"),
    col("siga"),
    col("backtracking_chegou"),
    from_json(col("AF_chegando"), af_chegando_schema).alias("AF_chegando"),
    col("tabela_origem"),
    col("data_carregamento")
)

print(f"* AF_chegando convertido para STRUCT")

In [0]:
# ============ PASSO 5: DESANINHAR AF_CHEGANDO ============
print("\n PASSO 5: Desaninhando AF_chegando...\n")

df_com_origem_destinos = df_recebidos_com_af.select(
    col("turno"),
    col("data"),
    col("agente"),
    col("voo"),
    col("iata"),
    col("AF_chegando.origem"),
    col("AF_chegando.destinos"),
    col("tabela_origem"),
    col("data_carregamento"),
)

print(f"* origem e destinos extraídos")

In [0]:
# ============ PASSO 6: EXPLODIR DESTINOS ============
print("\n PASSO 6: Explodindo destinos...\n")

df_destinos_explodido = df_com_origem_destinos.select(
    col("turno"),
    col("data"),
    col("agente"),
    col("voo"),
    col("iata"),
    col("origem"),
    explode_outer(col("destinos")).alias("destino_detail"),
    col("tabela_origem"),
    col("data_carregamento")
)

print(f"* {df_destinos_explodido.count()} registros após explodir destinos")

In [0]:
# ============ PASSO 7: DESANINHAR DESTINO_DETAIL ============
print("\n PASSO 7: Desaninhando destino_detail...\n")

df_tudo_flat = df_destinos_explodido.select(
    col("turno"),
    col("data"),
    col("agente"),
    col("voo"),
    col("iata"),
    col("origem"),
    col("destino_detail.voo").alias("destino_voo"),
    col("destino_detail.destino").alias("destino_iata"),
    col("destino_detail.qtd").alias("qtd"),
    col("tabela_origem"),
    col("data_carregamento")
)

print(f"* Tudo desaninhado e pronto")

In [0]:
from pyspark.sql.functions import regexp_extract, regexp_replace, when, col, lower

# Expressão regular para capturar 4 dígitos seguidos de 3 letras (com ou sem separadores)
padrao_voo = r"(\d{4})\s*[^a-zA-Z0-9]*\s*([a-zA-Z]{3})"

df_tudo_flat_corrigido = df_tudo_flat.withColumn(
    "turno",
    lower(col("turno"))
).withColumn(
    "voo",
    when(
        col("voo").rlike(padrao_voo),
        regexp_extract(col("voo"), padrao_voo, 1)
    ).otherwise(col("voo"))
).withColumn(
    "iata",
    when(
        col("voo").rlike(padrao_voo),
        regexp_extract(col("voo"), padrao_voo, 2)
    ).otherwise(col("iata"))
).withColumn(
    "origem",
    when(
        col("origem").rlike(padrao_voo),
        regexp_extract(col("origem"), padrao_voo, 1)
    ).otherwise(col("origem"))
).withColumn(
    "destino_voo",
    when(
        col("origem").rlike(padrao_voo),
        regexp_extract(col("origem"), padrao_voo, 2)
    ).otherwise(col("destino_voo"))
)

In [0]:
df_tudo_flat_renomeado = df_tudo_flat_corrigido.select(
    col("turno"),
    col("data"),
    col("agente"),
    col("voo").alias("arr_voo"),
    col("iata").alias("arr_iata"),
    col("origem").alias("af-origem"),
    col("destino_voo").alias("af_dep"),
    col("destino_iata").alias("af_iata_dp"),
    col("qtd").alias("af_qtd"),
    col("tabela_origem"),
    col("data_carregamento")
)

In [0]:
from pyspark.sql.functions import when, lit

df_tudo_flat_renomeado = df_tudo_flat_renomeado.withColumn(
    "af-origem",
    when(col("af-origem").isNull(), col("arr_voo")).otherwise(col("af-origem"))
).withColumn(
    "af_iata_dp",
    when(col("af_iata_dp").isNull(), lit("BSB")).otherwise(col("af_iata_dp"))
)

In [0]:
# ============ PASSO 8: CRIAR ID ÚNICO ============
print("\n PASSO 8: Criando ID único...\n")

df_com_id = df_tudo_flat_renomeado.select(
    concat_ws("_",
        regexp_replace(col("data"), "/", ""),
        col("agente"),
        concat(lit('o'), col("arr_voo")),
        col("arr_iata"),
        when(col("af_dep").isNull(), lit("")).otherwise(concat(lit('d'), col("af_dep"))),
        when(col("af_iata_dp").isNull(), lit("")).otherwise(concat(lit('d'), col("af_iata_dp")))
    ).alias("id_voo"),
    col("*")
)

print(f" ID criado: data_agente_voo_origem_destino")

In [0]:
# ============ PASSO 9: SELECIONAR, RENOMEAR E TRANSFORMAR ============
print("\n PASSO 9: Transformando (upper/lower)...\n")

df_silver_final = df_com_id.select(
    lower(col("id_voo")).alias("id_voos"),
    lower(col("turno")).alias("turno"),
    to_date(col("data"), "dd/MM/yyyy").alias("data"),
    upper(col("agente")).alias("agente"),
    upper(col("arr_voo")).alias("arr_voo"),
    upper(col("arr_iata")).alias("arr_iata"),
    upper(col("af-origem")).alias("origem_af"),
    upper(col("af_dep")).alias("destino_af"),
    upper(col("af_iata_dp")).alias("dep_iata_dep"),
    col("af_qtd").cast(IntegerType()).alias("arr_qtd"),
    col("tabela_origem"),
    col("data_carregamento")
)

print(f"* {df_silver_final.count()} registros finais")


In [0]:

# ============ PASSO 10: MERGE NA TABELA SILVER ============
print("Salvando na SILVER com MERGE...\n")

# Criar view temporária
df_silver_final.createOrReplaceTempView("temp_silver_arr")

try:
    # Tabela já existe - fazer MERGE
    spark.sql(f"""
        MERGE INTO {table_silver} AS target
        USING temp_silver_arr AS source
        ON target.id_voos = source.id_voos
        
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    print(f"✓ MERGE executado - apenas novos registros inseridos")
    
except:
    # Primeira execução - criar tabela
    df_silver_final.write.format("delta").mode("overwrite").saveAsTable(table_silver)
    print(f"* Tabela criada (primeira execução)")



In [0]:
# ============ PASSO 11: VERIFICAR RESULTADO ============
print("\nPASSO 11: Verificando resultado...\n")

df_silver = spark.table(table_silver)
print(f"✓ Total na SILVER: {df_silver.count()}")

print("\nÚltimos registros:")
df_silver.orderBy(col("data_carregamento").desc()).limit(3).display()

print("\n" + "="*70)
print("SILVER ATUALIZADA!")
print("="*70)